# Red vs Blue — Prisoner's Dilemma: Optimal Strategy Analysis

This notebook walks through the mathematics of the **infinitely-repeated Prisoner's Dilemma** and derives the analytical **decision boundary** that separates the cooperation region from the defection region.

## Outline
1. Stage-game setup and payoff matrix
2. Nash Equilibrium in the one-shot game
3. The repeated game and the Folk Theorem
4. Analytical decision boundary: δ\* = (T − R) / (T − P)
5. Strategy comparison via simulation
6. Decision boundary visualisation

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..') / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from red_vs_blue.model import (
    PayoffMatrix,
    compute_critical_delta,
    compute_decision_boundary,
    get_optimal_action,
    is_cooperation_sustainable,
)
from red_vs_blue.strategies import (
    Action, AlwaysCooperate, AlwaysDefect, TitForTat, GrimTrigger, simulate
)

## 1 · Stage-Game Payoff Matrix

| | Opponent **C** | Opponent **D** |
|---|---|---|
| **You C** | R (Reward) | S (Sucker) |
| **You D** | T (Temptation) | P (Punishment) |

Standard ordering: **T > R > P > S** and **2R > T + S**

In [ ]:
# Classic Axelrod parameters
matrix = PayoffMatrix(T=5.0, R=3.0, P=1.0, S=0.0)
print(f"Payoff matrix: T={matrix.T}, R={matrix.R}, P={matrix.P}, S={matrix.S}")
print(f"T > R > P > S: {matrix.T} > {matrix.R} > {matrix.P} > {matrix.S}  ✓")
print(f"2R > T+S: {2*matrix.R} > {matrix.T + matrix.S}  ✓")

## 2 · One-Shot Nash Equilibrium

In the single-stage game **Defect strictly dominates Cooperate** for both players.  The unique Nash Equilibrium is (D, D) even though (C, C) gives both players a higher payoff.

In [ ]:
print("One-shot best responses:")
for opp in ['C', 'D']:
    if opp == 'C':
        print(f"  Against C: play C → {matrix.R},  play D → {matrix.T}   ∴ best response = D")
    else:
        print(f"  Against D: play C → {matrix.S},  play D → {matrix.P}   ∴ best response = D")
print("\n→ Unique Nash Equilibrium: (D, D) with payoffs", (matrix.P, matrix.P))

## 3 · The Repeated Game — Folk Theorem

With discount factor δ ∈ (0,1), the discounted value of always cooperating (mutual C) is:

$$V_C = R + \delta R + \delta^2 R + \cdots = \frac{R}{1-\delta}$$

If you deviate (play D once) while the opponent uses Grim Trigger, you get T today then P forever:

$$V_D = T + \frac{\delta P}{1-\delta}$$

Cooperation is a Nash Equilibrium when $V_C \geq V_D$, yielding the decision boundary.

## 4 · Analytical Decision Boundary

In [ ]:
delta_star = compute_critical_delta(matrix)
print(f"δ* = (T - R) / (T - P) = ({matrix.T} - {matrix.R}) / ({matrix.T} - {matrix.P}) = {delta_star:.4f}")
print()
for delta in [0.3, 0.5, 0.75, 0.9]:
    action = get_optimal_action(matrix, delta)
    marker = '✓' if action == 'Cooperate' else '✗'
    print(f"  δ = {delta:.2f}  →  {action:10s}  {marker}  (δ {'≥' if delta >= delta_star else '<'} δ*)")

## 5 · Strategy Comparison

In [ ]:
strategies = [AlwaysCooperate(), AlwaysDefect(), TitForTat(), GrimTrigger()]
delta = 0.9
rounds = 200

print(f"Discounted payoffs (δ={delta}, rounds={rounds})")
print(f"{'':20s}  " + "  ".join(f"{s.name:16s}" for s in strategies))
for s1 in strategies:
    row = []
    for s2 in strategies:
        p1, _ = simulate(s1, s2, matrix, rounds=rounds, delta=delta)
        row.append(f"{p1:16.2f}")
    print(f"{s1.name:20s}  " + "  ".join(row))

## 6 · Decision Boundary Visualisation

In [ ]:
T_grid, delta_grid, coop_mask = compute_decision_boundary(
    matrix, t_range=(3.05, 10.0), delta_range=(0.01, 0.99), resolution=200
)

# Analytical boundary
t_curve = np.linspace(3.05, 10.0, 500)
delta_curve = (t_curve - matrix.R) / (t_curve - matrix.P)

fig, ax = plt.subplots(figsize=(9, 6))
cmap = mcolors.ListedColormap(['#c0392b', '#2980b9'])
ax.pcolormesh(T_grid, delta_grid, coop_mask.astype(int), cmap=cmap, alpha=0.45, shading='auto')
ax.plot(t_curve, delta_curve, 'w--', lw=2, label='Decision boundary δ*')

# Mark current example state
example_T, example_delta = 5.0, 0.8
ax.scatter([example_T], [example_delta], marker='*', s=250, color='gold',
           zorder=5, label=f'Example state (T={example_T}, δ={example_delta})')

ax.text(9.2, 0.88, 'COOPERATE', color='#74b9ff', fontsize=13, ha='right')
ax.text(9.2, 0.08, 'DEFECT',    color='#ff7675', fontsize=13, ha='right')

ax.set_xlabel('T — Temptation', fontsize=12)
ax.set_ylabel('δ — Discount Factor', fontsize=12)
ax.set_title("Decision Boundary: Prisoner's Dilemma Optimal Strategy", fontsize=13)
ax.legend(loc='upper right')
ax.set_facecolor('#0e1117')
fig.patch.set_facecolor('#0e1117')
for spine in ax.spines.values():
    spine.set_edgecolor('#555')
ax.tick_params(colors='white')
ax.yaxis.label.set_color('white')
ax.xaxis.label.set_color('white')
ax.title.set_color('white')
plt.tight_layout()
plt.savefig('decision_boundary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved decision_boundary.png')